In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from modules.coord_transform import *
from modules.simulation import *
from modules.likelihood import *
from modules.mcmc import run_mcmc  
from modules.localise import ci_68_and_sigma
from modules.constellation import get_satellite_positions,generate_full_constellation,get_pointing_radec,eci_to_latlon


import os
from matplotlib.patches import Patch

In [ ]:
def catalogue(type):
    if type =="short":
        cat = pd.read_csv("D:\\Unimelb\\SpIRIT_In_Polar_Orbit\\mcmc\\others\\short_grb_catalogue_filter.txt",sep ='|')
    elif type=="long":
        cat = pd.read_csv("D:\\Unimelb\\SpIRIT_In_Polar_Orbit\\mcmc\\others\\long_grb_catalogue_filter.txt", sep='|' )
    return cat

In [ ]:
rng_grb = np.random.default_rng(seed=4)
type ="long"
cat = catalogue(type)
precomputed_grbs = []
while len(precomputed_grbs) < 1000:  # or however many you need
    grb_vec, ra, dec, t_90, flux_avg = generate_grb(cat, rng_grb)
    time = rng_grb.uniform(0, 96 * 60)
    precomputed_grbs.append((grb_vec, ra, dec, t_90, flux_avg,time))

offset_angle = [10]
planes = [
        #{"incl": 0, "rot": 0},       # Equatorial
       {"incl": 100, "rot": 0},     # Polar
        #{"incl": 50, "rot": 0},      # Diagonal 1 
        #{"incl": 150, "rot": 0},     # Diagonal 2 
    ]

flux_limit = flux_lower_bound(type) # Change flux_limit for long grb = 0.463 and for short grb= 1.861
minimum_num_sats_detecting_burst = 3

df_results_ = {}
for i, offset in enumerate(offset_angle):
    rng = np.random.default_rng(seed=2)
    Area = 50
    localisation_result = []
    n_det = 0
    total_generate = 0  # Counter for total GRBs generated
    for grb_idx, (grb_vec, ra, dec, t_90, flux_avg,time) in enumerate(precomputed_grbs):
        if n_det >= 1:
            break
        print(time)
        sat_pos = generate_full_constellation(time, planes)
        lat_lon = eci_to_latlon(sat_pos, time)
        num_sats = len(sat_pos)

        pointings, base_dirs = get_pointing_radec(sat_pos, num_sats , offset_deg = offset_angle)
        sat_pointing = np.array([coord_transform.r2c(ra, dec) for ra, dec in pointings])

        print(ra,dec)
        
        grb_info = {"ra": ra, 
                "dec":dec, 
                "flux_avg":flux_avg, 
                "t_90": t_90,
                "rng":rng}  
        det_result = simulate_satellite_det(grb_vec, flux_avg, sat_pos, sat_pointing, flux_limit, lat_lon)

        sat_info = {'Area': Area, 
                    'sat_pos' : sat_pos,
                'sat_pointing': sat_pointing, 
                'flux_limit': flux_limit,
                'offset': offset,
                'lat_lon':lat_lon}

        if sum( det_result[0] > 0 ) >=  minimum_num_sats_detecting_burst :  # Ensure it skips GRBs that do not meet conditions
        
            print(ra, dec)
            n_det += 1
            f_obs, t_obs= det_result
            total_generate = grb_idx + 1


            Ph_obs = np.array([ 0 if f < flux_limit 
                            else rng.poisson(f * t_90 * Area) for f in f_obs])  # Observed Photon count
            
            save_path = os.path.join( f"./orbit_results/{num_sats}_90c_{type}_grb_plots", f"offset_{offset}", f"detection_{n_det}")
            
            obs_info = {'t_obs': t_obs,
                    'f_obs': f_obs,
                    'Ph_obs': Ph_obs }
        
            mcmc_params= {'steps': 5000, 
                      'nwalk': 8, 
                      'discard': 1000,
                      'move': 2.5, 
                      'save_path': save_path,
                      'corner_show' :False ,'corner_save' : True, 
                      'chain_show': False, 'chain_save' : True}
            
            #localise_params ={'localisation_show' :False, 'localisation_save' :True}
        
            flat_samples = run_mcmc(**grb_info,
                                        **obs_info,
                                        **sat_info,
                                        **mcmc_params)
            
            true_value = np.array([ra, dec, flux_avg])
            ci_area_68, containment, resolution = ci_68_and_sigma(flat_samples, true_value,
                                                   type,
                                                   save_path,
                                                   localisation_show=False,
                                                   localisation_save=True)
                                                        

            print(f'RA:{ra},DEC:{dec},FLUX:{flux_avg}')
            print(f"68% confidence region area: {ci_area_68:.2f} deg²")
            print(f"area of grid:{resolution}deg²")
            print(f'Localization completed for detection #{n_det}')    
        
            localisation_result.append([ra, dec, t_90, flux_avg, time, f_obs, np.count_nonzero(f_obs), ci_area_68, containment, total_generate])

        df_results_[i] = pd.DataFrame(localisation_result, columns=['RA', 'Dec', 'T90', 'Flux_Avg', 'Time', 'fcos','n_det' ,'CI_Area_68', 'containment','Total generated'])
    
df_results_[i].to_csv(f'./orbit_results/{num_sats}_90c_{type}_grb_plots/{type}_{offset}.csv', index=False)
print(f'Total GRBs generated: {total_generate}, Total detected: {n_det}')

# Particle Background Map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the background data
combined_bkgd = np.load('particle_background.npy')

# Define the lat/lon grid
latitudes = np.arange(-90, 90)  # Assuming 1-degree resolution
longitudes = np.arange(-180, 180)

# Plot the background map
plt.figure(figsize=(10, 5))
plt.imshow(combined_bkgd, extent=[-180, 180, -90, 90], origin='lower', cmap='inferno', aspect='auto')
plt.colorbar(label="Particle Background Intensity")  # Add colorbar
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Particle Background Map")
plt.show()


# W.Leone Sigma_CC vs Flux

In [ ]:
df = pd.read_csv("C:\\Users\\Haritha\\Desktop\\CCFdata.csv")
dfl = df[df['t90'] < 2]
dfl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit



# Example data (replace with your own)
x = dfl['flux']
y = dfl['sigmacc']

# Power-law function
def power_law(x, a, b):
    return a * x**b

# Fit the data
params, covariance = curve_fit(power_law, x, y)
a, b = params

# Display the equation
print(f"Fitted equation: y = {a:.3f} * x^{b:.3f}")

# Plot
x_fit = np.linspace(min(x), max(x), 100)
y_fit = power_law(x_fit, a, b)

plt.scatter(x, y, label='Data')
plt.plot(x_fit, y_fit, 'r-', label=f'Fit: y = {a:.2f} * x^{b:.2f}')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Example data
x = dfl['flux'].values
y = dfl['sigmacc'].values

# Power-law function
def power_law(x, a, b):
    return a * x**b

# Fit the data
params, _ = curve_fit(power_law, x, y)
a, b = params
log_a = np.log10(a)

# Fit line
x_fit = np.linspace(min(x), max(x), 100)
y_fit = power_law(x_fit, a, b)

# Equation as legend label (LaTeX formatted)
equation_label = rf'$\log(\sigma_{{cc}}) = {log_a:.2f} + {b:.2f} \cdot \log(F)$'

# Plot
plt.figure(figsize=(8,6))
plt.scatter(x, y, label='W. Leone + 2025')
plt.plot(x_fit, y_fit, 'r-', label=equation_label)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Average Flux 50-300kev [ph/$cm^{2}$/s]')
plt.ylabel(r'$\sigma_{cc}$ (s) ')
#plt.grid(True, which='both', linestyle='--')
plt.legend(fontsize=12)
plt.title('Long GRBs')
plt.show()
print(f"Fitted log-log equation:")
print(f"log(sigmacc) = {log_a:.3f} + {b:.3f} * log(flux)")

# Plots

## Cumulative Fraction of Detections VS Localisation Unceratinity(deg2)

In [ ]:
angles = [ 0, 10, 15, 20, 30]  
colour = ['red', 'g','b', 'm', 'cyan', 'k', 'y']

# Initialize lists
deg_list = []
sorted_ci_areas_list = []
cdf_list = []

plt.figure(figsize=(10, 6))
plt.xscale('log')
#plt.ylim([0, 0.2]) 

for idx, i in enumerate(angles):
    file_path = f"./orbit_results/8_100_b3_long_grb_plots/long_{i}.csv"
    
    
    deg = pd.read_csv(file_path)
    deg_list.append(deg)

    
    sorted_ci_areas = np.sort(deg['CI_Area_68'])
    sorted_ci_areas_list.append(sorted_ci_areas)

    total_generated = deg['Total generated'].iloc[-1]
    cdf = np.arange(1, len(sorted_ci_areas) + 1) / total_generated
    cdf_list.append(cdf)

    # Plot 
    plt.plot(sorted_ci_areas, cdf, marker='o', linestyle='-', 
             color=colour[idx], markersize=1, label=f'{i} deg', alpha=1)

plt.xlabel("Credible Interval (CI) Area (deg²)")
plt.ylabel("Cumulative Probability")
plt.title("CDF of 68% CI area for Long GRB Localization with 8 polar satellites")
plt.legend()

#plt.savefig('./orbit_results/8_90_long_grb_plots/long_grb_8satpolarmoreangles.png')
plt.show()

## Area plot

In [ ]:
Area = [90,100,150,200]  
colour = ['red', 'g','b', 'm', 'cyan', 'k', 'y']

# Initialize lists
deg_list = []
sorted_ci_areas_list = []
cdf_list = []

plt.figure(figsize=(10, 6))
plt.xscale('log')
#plt.ylim([0, 0.2]) 

for idx, i in enumerate(Area):
    file_path = f"./orbit_results/8_{i}_long_grb_plots/long_15.csv"
    
    
    deg = pd.read_csv(file_path)
    deg_list.append(deg)

    
    sorted_ci_areas = np.sort(deg['CI_Area_68'])
    sorted_ci_areas_list.append(sorted_ci_areas)

    total_generated = deg['Total generated'].iloc[-1]
    cdf = np.arange(1, len(sorted_ci_areas) + 1) / total_generated
    cdf_list.append(cdf)

    # Plot 
    plt.plot(sorted_ci_areas, cdf, marker='o', linestyle='-', 
             color=colour[idx], markersize=1, label=f'{i} cmsq', alpha=1)

plt.xlabel("Credible Interval (CI) Area (deg²)")
plt.ylabel("Cumulative Probability")
plt.title("CDF of 68% CI area for Long GRB Localization with 8 polar satellites, 15 offset with increasing Area")
plt.legend()

#plt.savefig('./orbit_results/8_90_long_grb_plots/long_grb_8satpolarArea.png')
plt.show()


# With Centre and without centre comparison

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Define angles and colors for plotting
angles = [0,10,15]
colour = ['red', 'g', 'b', 'm', 'cyan', 'k', 'y']

# Initialize lists
deg_list = []
sorted_ci_areas_list = []
cdf_list = []

# Create a new figure
plt.figure(figsize=(10, 6))
plt.xscale('log')

# Plot for the first set of data
for idx, i in enumerate(angles):
    file_path = f"./orbit_results/8_90_long_grb_plots/long_{i}.csv"
    
    deg = pd.read_csv(file_path)
    deg_list.append(deg)

    sorted_ci_areas = np.sort(deg['CI_Area_68'])
    sorted_ci_areas_list.append(sorted_ci_areas)

    total_generated = deg['Total generated'].iloc[-1]
    cdf = np.arange(1, len(sorted_ci_areas) + 1) / total_generated
    cdf_list.append(cdf)

    # Plot first dataset
    plt.plot(sorted_ci_areas, cdf, marker='o', linestyle='-.', 
             color=colour[idx], markersize=1, label=f'{i} deg - without c ', alpha=1)

# Plot for the second set of data
for idx, i in enumerate(angles):
    file_path = f"./orbit_results/8_90c_long_grb_plots/long_{i}.csv"
    
    deg = pd.read_csv(file_path)
    deg_list.append(deg)

    sorted_ci_areas = np.sort(deg['CI_Area_68'])
    sorted_ci_areas_list.append(sorted_ci_areas)

    total_generated = deg['Total generated'].iloc[-1]
    cdf = np.arange(1, len(sorted_ci_areas) + 1) / total_generated
    cdf_list.append(cdf)

    # Plot second dataset
    plt.plot(sorted_ci_areas, cdf, marker='o', linestyle='-', 
             color=colour[idx + len(angles)], markersize=1, label=f'{i} deg withc', alpha=1)
    plt.xlabel("Credible Interval (CI) Area (deg²)")
plt.ylabel("Cumulative Probability")
plt.title("CDF of 68% CI area for Long GRB Localization with satellites in Polar orbit")
plt.legend(loc='upper left')